In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import csv
import time
import requests
from bs4 import BeautifulSoup
import re
import json
from urllib.parse import urlparse, parse_qs
import time

headers = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win 64; x64) AppleWebKit/537.36 (KHTML, Like Gecko) Chrome/140.0.0.0 Safari/537.36 Edg/140.0.0.0'
}

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

In [3]:
urlokezone = 'https://search.okezone.com/search?q=politik&highlight=1&sort=desc&start=0'
resokezone = requests.get(urlokezone, headers=headers)

soupokezone = BeautifulSoup(resokezone.text, 'lxml')
# print(soupokezone)

In [4]:
def checklastpage(url):
    lastpage = -1
    # Setup Chrome
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # jalan background
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")


    driver = webdriver.Chrome(options=chrome_options)

    try:
        driver.get(url)
        
        wait = WebDriverWait(driver, 10)
        
        try:
            pagination_elements = wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "a")))
            
            loaddata_links = []
            for element in pagination_elements:
                href = element.get_attribute('href')
                if href and 'loaddata' in href:
                    loaddata_links.append({
                        'href': href,
                        'text': element.text.strip()
                    })
            
            print(f"Found {len(loaddata_links)} loaddata links:")
            for link in loaddata_links:
                print(f"Link: {link['href']} - Text: '{link['text']}'")
            
            last_links = [link for link in loaddata_links if link['text'].lower() == 'last']
            
            if last_links:
                last_link = last_links[0]
                href = last_link['href']
                print(f"\nFound Last link: {href}")
                
                match = re.search(r'/(\d+)/?$', href)
                if match:
                    lastpage = int(match.group(1))
                    print(f"Last page number: {lastpage}")
                else:
                    print("Could not extract page number")
            else:
                print("No Last link found")
                
        except Exception as e:
            print(f"Error waiting for pagination: {e}")
            
            page_source = driver.page_source
            soup_selenium = BeautifulSoup(page_source, 'lxml')
            
            loaddata_links = soup_selenium.find_all('a', href=lambda x: x and 'loaddata' in x)
            print(f"\nFrom page source - Found {len(loaddata_links)} loaddata links:")
            
            for link in loaddata_links[:5]:
                print(f"Link: {link.get('href')} - Text: '{link.get_text().strip()}'")

    finally:
        driver.quit()
    return lastpage

In [5]:
totalpage = checklastpage(urlokezone)

Found 18 loaddata links:
Link: https://search.okezone.com/loaddata/article/politik/2 - Text: '2'
Link: https://search.okezone.com/loaddata/article/politik/3 - Text: '3'
Link: https://search.okezone.com/loaddata/article/politik/4 - Text: '4'
Link: https://search.okezone.com/loaddata/article/politik/5 - Text: '5'
Link: https://search.okezone.com/loaddata/article/politik/2 - Text: '»'
Link: https://search.okezone.com/loaddata/article/politik/10 - Text: 'Last'
Link: https://search.okezone.com/loaddata/foto/politik/2 - Text: ''
Link: https://search.okezone.com/loaddata/foto/politik/3 - Text: ''
Link: https://search.okezone.com/loaddata/foto/politik/4 - Text: ''
Link: https://search.okezone.com/loaddata/foto/politik/5 - Text: ''
Link: https://search.okezone.com/loaddata/foto/politik/2 - Text: ''
Link: https://search.okezone.com/loaddata/foto/politik/10 - Text: ''
Link: https://search.okezone.com/loaddata/video/politik/2 - Text: ''
Link: https://search.okezone.com/loaddata/video/politik/3 - T

In [6]:
print(f"page yg bisa di scrap: {totalpage}")

page yg bisa di scrap: 10


In [7]:
# baca link berita
def getarticle(link):
  try:
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    div_content = soup.find('div', class_='c-detail read') #ganti class sesuai nama class di artikel
    paragraphs = div_content.find_all('p') #ganti tiap tag paragraf
    content = ' '.join([p.get_text(strip=True) for p in paragraphs])
    return content
    
  except Exception as e:
    print(f"Error baca artikel di: {link} | {e}")
  return ''


In [28]:
# --- Konfigurasi dan Setup ---
CSV_FILE = 'okezone_politik_articles.csv'
CSV_HEADERS = ['title', 'tanggal', 'waktu', 'content']

# --- Fungsi Inisialisasi CSV ---
# def initialize_csv(filename, headers):
#     """Membuat file CSV baru dan menulis header."""
#     try:
#         with open(filename, 'w', newline='', encoding='utf-8') as f:
#             writer = csv.writer(f)
#             writer.writerow(headers)
#         print(f"✅ CSV file '{filename}' initialized with headers: {headers}")
#     except Exception as e:
#         print(f"Error initializing CSV file: {e}")


# --- Fungsi Penulisan Akhir CSV ---
def write_data_to_csv(filename, headers, data):
    """Menulis seluruh data yang dikumpulkan ke file CSV."""
    try:
        # Menggunakan mode 'w' (write) karena file ditulis hanya sekali
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(headers) # Tulis header
            writer.writerows(data)  # Tulis semua data sekaligus
        print(f"\n✅ SUCCESS: Total {len(data)} articles successfully saved to '{filename}'")
    except Exception as e:
        print(f"\n❌ FATAL ERROR: Failed to write data to CSV file '{filename}'. Error: {e}")

In [ ]:
#SCRAPING
scraped_data = []
counter = 0

#coba 2 halaman aj
for page in range(1, totalpage+1):
    print(f'\n--- Scraping page: {page} ---')
    url = f'https://search.okezone.com/loaddata/article/politik/{page}'
    try:
        res = requests.get(url, headers=headers)
        # if res.status_code != 200 or not res.text.strip():
        #     print(f"Failed to fetch page {page}. Status: {res.status_code}")
        #     continue

        soup = BeautifulSoup(res.text, 'lxml')
        articles = soup.find_all('div', class_='subgroup')

        if not articles:
             print(f"No articles found on page {page}. Stopping.")
             break

        with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)

        for art in articles:
            desc = art.select_one('div', class_='desc_section')
            try:
                title = desc.select_one('a').text.strip()
                link = desc.select_one('a', class_='desc-text')['href']

                #ambil tanggal
                temp = desc.find('a', class_='time-text')
                tanggal_waktu = temp.text.strip()
                tanggal = tanggal_waktu.split(' ')[0] + ' ' + tanggal_waktu.split(' ')[1] + ' ' + tanggal_waktu.split(' ')[2] #tanggalnya
                waktu = tanggal_waktu.split(' ')[3] + ' ' + tanggal_waktu.split(' ')[4] #jamnya


                content = getarticle(link)

                # Simpan data ke list, nanti disimpan CSV secara langsung
                scraped_data.append([title, tanggal, waktu, content])
                
                print(f'✅ Article #{counter+1} SCRAPPED! Title: {title[:50]}...')
                print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]} \n Waktu: {tanggal}')
                counter += 1
                time.sleep(1)
            except Exception as e:
                print(f"Error parsing artikel di hlman: {page} | {e}")
                continue
    except Exception as e:
        print(f"Error scraping di hlman: {page} | {e}")
        continue


print(f"\n--- SCRAPING LOGIC SELESAI ({counter} articles collected) ---")
write_data_to_csv(CSV_FILE, CSV_HEADERS, scraped_data)


--- Scraping page: 1 ---
✅ Article #1 SCRAPPED! Title: Pak Bas Lapor Progres IKN ke Istana, Persiapan Ibu...
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA- Kepala Otorita Ibu Kota Nusantara (OIKN) Basuki Hadimuljono merapat ke Kementerian Sekretar 
 Waktu: 03 Oktober 2025
✅ Article #2 SCRAPPED! Title: Fenomena Selebriti di Politik, Begini Pandangan Su...
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA- Dalam episode terbaru Bisikan Gaib, Robby Purba menghadirkan tamu spesial, Ki Atmo, yang di 
 Waktu: 01 Oktober 2025
✅ Article #3 SCRAPPED! Title: SAS Institute: Program MBG Bukan Janji Politik, Ta...
    100 KARAKTER PERTAMA DI CONTENT:  JAKARTA– Badan Gizi Nasional (BGN) mencatat hingga 22 September 2025, terdapat 4.711 kasus bakteri  
 Waktu: 01 Oktober 2025
✅ Article #4 SCRAPPED! Title: Husnan Bey: PPP Harus Kembali ke Khitah, Jangan Ja...
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA- Muktamar X Partai Persatuan Pembangunan (PPP) yang baru saja berakhir memunculkan dualisme  
 Waktu: 30